In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_inventory=spark.read.format("delta").load("s3://retail-lakehouse-ashu/bronze/inventory/")

In [0]:
bronze_inventory.printSchema()

In [0]:
bronze_inventory_explode=bronze_inventory.withColumn("products",explode_outer(col("products")))

In [0]:
bronze_inventory_explode.groupBy(col("warehouse_id")).count().filter(col("count")>1).display()

In [0]:
bronze_inventory_explode.groupBy("warehouse_id").agg(
    countDistinct("warehouse_name").alias("warehouse_name_count")
).filter(
    col("warehouse_name_count") > 1
).show()

In [0]:
bronze_inventory_explode.groupBy(
    "warehouse_id",
    "products.product_id"
).count().filter(
    col("count") > 1
).show()

In [0]:
dim_warehouse=bronze_inventory_explode.select(col("warehouse_id"),col("warehouse_name")).dropDuplicates(["warehouse_id"])
dim_warehouse.display()

In [0]:
bronze_inventory_explode.display()

In [0]:
fact_inventory=bronze_inventory_explode.select(col("products.available_stock").alias("available_stock"),col("products.product_id").alias("product_id"),col("products.reorder_level").alias("reorder_level"),col("warehouse_id"),col("ingestion_timestamp"),col("source_system"),col("batch_id"),col("source_file"))
fact_inventory.display()

In [0]:
input_inventory=bronze_inventory.count()
output_warehouse=dim_warehouse.count()
output_inventory=fact_inventory.count()

print("Input Records:", input_inventory)
print("Warehouse Records:", output_warehouse)
print("Inventory Records:", output_inventory)

In [0]:
dim_warehouse.write.mode("overwrite").format("delta").save("s3://retail-lakehouse-ashu/silver/inventory/dim_warehouse")
fact_inventory.write.mode("overwrite").format("delta").save("s3://retail-lakehouse-ashu/silver/inventory/fact_inventory")